# JOBSHEET 04: SEGMENTASI GAMBAR
## Praktikum Pengolahan Citra
**Politeknik Negeri Semarang – Jurusan Teknik Elektro**  
**Program Studi STR Teknologi Rekayasa Komputer**

**Dosen Pengampu:** Ir. Prayitno, S.ST., M.T., Ph.D.

---
- **Nama Mahasiswa:** _(isi nama kamu)_
- **NIM:** _(isi NIM kamu)_
- **Kelompok:** _(isi kelompok kamu)_
- **Semester:** Genap 2025

## Instalasi Library (jalankan jika belum terinstall)

In [ ]:
# Jalankan cell ini jika library belum terinstall
# !pip install scikit-image matplotlib numpy scikit-learn scipy

---
## Praktikum 1: Segmentasi Menggunakan Thresholding Global dan Otsu

**Tujuan:** Menerapkan metode segmentasi berbasis thresholding pada citra grayscale dan membandingkan hasil thresholding manual (global) dengan metode Otsu.

In [ ]:
import matplotlib.pyplot as plt
from skimage import data, filters, img_as_ubyte
from skimage.color import rgb2gray
import numpy as np

# 1. Memuat citra (contoh: coins)
image_coins = data.coins()  # Citra sudah grayscale

print("=" * 55)
print("   PRAKTIKUM 1 – Thresholding Global dan Otsu")
print("=" * 55)
print(f"[INFO] Citra dimuat: data.coins()")
print(f"[INFO] Ukuran citra     : {image_coins.shape} piksel")
print(f"[INFO] Tipe data        : {image_coins.dtype}")
print(f"[INFO] Nilai min piksel : {image_coins.min()}")
print(f"[INFO] Nilai max piksel : {image_coins.max()}")
print(f"[INFO] Nilai rata-rata  : {image_coins.mean():.2f}")
print("-" * 55)

# 2. Thresholding Global (manual)
thresh_manual = 100
binary_manual = image_coins > thresh_manual
pixels_putih_manual = binary_manual.sum()
pixels_hitam_manual = (~binary_manual).sum()
print(f"[MANUAL] Nilai threshold yang digunakan : T = {thresh_manual}")
print(f"[MANUAL] Piksel objek (putih/True)      : {pixels_putih_manual} piksel")
print(f"[MANUAL] Piksel background (hitam/False): {pixels_hitam_manual} piksel")
print("-" * 55)

# 3. Thresholding Otsu
thresh_otsu = filters.threshold_otsu(image_coins)
binary_otsu = image_coins > thresh_otsu
pixels_putih_otsu = binary_otsu.sum()
pixels_hitam_otsu = (~binary_otsu).sum()
print(f"[OTSU]   Nilai threshold otomatis Otsu  : T = {thresh_otsu:.2f}")
print(f"[OTSU]   Piksel objek (putih/True)      : {pixels_putih_otsu} piksel")
print(f"[OTSU]   Piksel background (hitam/False): {pixels_hitam_otsu} piksel")
print("-" * 55)
print(f"[INFO] Selisih nilai threshold           : {abs(thresh_otsu - thresh_manual):.2f}")
print(f"[INFO] Otsu lebih {'tinggi' if thresh_otsu > thresh_manual else 'rendah'} dari threshold manual")
print("=" * 55)

# 4. Visualisasi Hasil
fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image_coins, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(binary_manual, cmap=plt.cm.gray)
ax[1].set_title(f'Manual Threshold (T={thresh_manual})')
ax[1].axis('off')

ax[2].imshow(binary_otsu, cmap=plt.cm.gray)
ax[2].set_title(f"Otsu's Threshold (T={thresh_otsu:.2f})")
ax[2].axis('off')

plt.tight_layout()
plt.show()

### Analisis Hasil Praktikum 1

**Pertanyaan:** Amati perbedaan hasil segmentasi antara thresholding manual dan Otsu. Apakah metode Otsu berhasil memisahkan objek (koin) dari latar belakang dengan baik pada citra ini?

**Jawaban:**

Dari hasil eksperimen di atas, dapat diamati perbedaan yang jelas antara dua metode thresholding:

- **Thresholding Manual (T=100):** Nilai ambang ditentukan secara manual tanpa mempertimbangkan distribusi intensitas histogram citra. Akibatnya, banyak piksel latar belakang ikut terklasifikasi sebagai objek (false positive), sehingga batas koin kurang bersih dan terdapat banyak noise putih di area background.

- **Metode Otsu (T≈107):** Otsu secara otomatis menentukan nilai ambang optimal dengan memaksimalkan varians *antar-kelas* (between-class variance). Hasilnya jauh lebih bersih — koin-koin terpisah dengan jelas dari latar belakang dengan bentuk yang lebih bulat dan presisi.

**Kesimpulan:** Ya, metode Otsu berhasil memisahkan objek (koin) dari latar belakang dengan lebih baik dibandingkan threshold manual. Hal ini karena Otsu secara adaptif mencari nilai pemisah terbaik berdasarkan histogram citra, bukan tebakan nilai tetap. Citra `data.coins()` memiliki histogram yang bersifat bimodal (dua puncak jelas untuk koin dan background), sehingga Otsu sangat efektif di sini.

---
## Praktikum 2: Segmentasi Menggunakan Region Growing (Contoh Sederhana)

**Tujuan:** Mendemonstrasikan konsep dasar region growing menggunakan fungsi `flood` dari `skimage.segmentation`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, segmentation

image_camera = data.camera()

print("=" * 55)
print("   PRAKTIKUM 2 – Region Growing (Flood Fill)")
print("=" * 55)
print(f"[INFO] Citra dimuat: data.camera()")
print(f"[INFO] Ukuran citra : {image_camera.shape} piksel")
print(f"[INFO] Total piksel : {image_camera.size}")
print("-" * 55)

# Seed point
seed_point = (50, 150)
intensitas_seed = image_camera[seed_point[0], seed_point[1]]
print(f"[SEED]  Koordinat seed point : y={seed_point[0]}, x={seed_point[1]}")
print(f"[SEED]  Nilai intensitas seed : {intensitas_seed}")
print("-" * 55)

# Flood fill dengan tolerance=10
tolerance = 10
flood_mask = segmentation.flood(image_camera, seed_point, tolerance=tolerance)
region_size = flood_mask.sum()
persen_region = (region_size / image_camera.size) * 100

print(f"[FLOOD] Nilai tolerance        : {tolerance}")
print(f"[FLOOD] Rentang intensitas yang diterima: [{intensitas_seed - tolerance} – {intensitas_seed + tolerance}]")
print(f"[FLOOD] Jumlah piksel dalam region : {region_size} piksel")
print(f"[FLOOD] Persentase area region     : {persen_region:.2f}% dari total citra")
print("=" * 55)

# Buat citra tersegmentasi
segmented_image = np.copy(image_camera)
segmented_image[flood_mask] = 255

fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image_camera, cmap=plt.cm.gray)
ax[0].plot(seed_point[1], seed_point[0], 'ro', markersize=8, label='Seed')
ax[0].set_title(f'Original Image\n(Seed: y={seed_point[0]}, x={seed_point[1]})')
ax[0].legend(loc='upper right')
ax[0].axis('off')

ax[1].imshow(flood_mask, cmap=plt.cm.gray)
ax[1].set_title(f'Flood Fill Mask\n(tolerance={tolerance}, {region_size} piksel)')
ax[1].axis('off')

ax[2].imshow(segmented_image, cmap=plt.cm.gray)
ax[2].set_title('Segmented Image\n(Region ditandai putih)')
ax[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# === EKSPERIMEN: Variasi Seed Point dan Tolerance ===
print("=" * 60)
print("   EKSPERIMEN – Variasi Seed & Tolerance")
print("=" * 60)
print(f"{'No':<4} {'Seed (y,x)':<14} {'Tolerance':<12} {'Piksel Region':<16} {'% Area'}")
print("-" * 60)

eksperimen = [
    {'seed': (50, 150),  'tolerance': 5,  'label': 'tol=5  (ketat)'},
    {'seed': (50, 150),  'tolerance': 15, 'label': 'tol=15 (sedang)'},
    {'seed': (50, 150),  'tolerance': 30, 'label': 'tol=30 (longgar)'},
    {'seed': (200, 100), 'tolerance': 10, 'label': 'seed area gelap'},
    {'seed': (300, 200), 'tolerance': 10, 'label': 'seed area tengah'},
    {'seed': (100, 300), 'tolerance': 20, 'label': 'seed lain, tol=20'},
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, exp in enumerate(eksperimen):
    mask = segmentation.flood(image_camera, exp['seed'], tolerance=exp['tolerance'])
    result = np.copy(image_camera)
    result[mask] = 255
    n_piksel = mask.sum()
    pct = (n_piksel / image_camera.size) * 100

    print(f"{idx+1:<4} {str(exp['seed']):<14} {exp['tolerance']:<12} {n_piksel:<16} {pct:.2f}%  → {exp['label']}")

    row, col = divmod(idx, 3)
    axes[row][col].imshow(result, cmap=plt.cm.gray)
    axes[row][col].plot(exp['seed'][1], exp['seed'][0], 'ro', markersize=8)
    axes[row][col].set_title(f"Seed{exp['seed']} tol={exp['tolerance']}\n({n_piksel} piksel, {pct:.1f}%)", fontsize=9)
    axes[row][col].axis('off')

print("=" * 60)

plt.suptitle('Eksperimen Region Growing – Variasi Seed & Tolerance', fontsize=12)
plt.tight_layout()
plt.show()

### Analisis Hasil Praktikum 2

**Pertanyaan:** Bagaimana pengaruh parameter `tolerance` terhadap ukuran region yang 'tumbuh'? Cobalah mengubah titik seed dan nilai tolerance.

**Jawaban:**

Berdasarkan hasil eksperimen yang terlihat pada tabel output di atas, dapat disimpulkan beberapa hal:

1. **Pengaruh `tolerance` terhadap ukuran region:**
   Semakin besar nilai `tolerance`, semakin luas region yang tumbuh. Hal ini karena algoritma flood fill akan menerima piksel tetangga yang nilai intensitasnya lebih jauh dari seed — sehingga region "menjalar" ke area yang lebih beragam. Sebaliknya, tolerance kecil (misalnya 5) menghasilkan region yang sangat kecil dan ketat karena hanya piksel dengan intensitas sangat mirip seed yang diterima.

2. **Pengaruh pemilihan seed point:**
   Titik seed yang berada di area homogen (seperti langit/background terang) akan menghasilkan region yang besar dan melebar ke seluruh area serupa. Sebaliknya, seed di area objek gelap (seperti badan kameraman) menghasilkan region lebih kecil dan terbatas di dalam objek tersebut.

3. **Keterbatasan metode:**
   Region growing sangat sensitif terhadap pemilihan seed dan nilai tolerance. Nilai tolerance yang terlalu besar menyebabkan *over-segmentation* (region terlalu besar), sementara terlalu kecil mengakibatkan *under-segmentation* (region terlalu kecil dan tidak merepresentasikan objek sebenarnya).

---
## Praktikum 3: Segmentasi Citra Berwarna Menggunakan K-Means Clustering

**Tujuan:** Menerapkan K-Means clustering untuk mensegmentasi citra berwarna berdasarkan nilai warna piksel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from sklearn.cluster import KMeans
from skimage.color import rgb2lab, lab2rgb
import warnings

image_astro = data.astronaut()
image_astro_float = image_astro.astype(float) / 255.0

print("=" * 58)
print("   PRAKTIKUM 3 – K-Means Clustering")
print("=" * 58)
print(f"[INFO] Citra dimuat       : data.astronaut()")
print(f"[INFO] Ukuran citra       : {image_astro.shape} (H x W x Channels)")
print(f"[INFO] Total piksel       : {image_astro.shape[0] * image_astro.shape[1]}")
print(f"[INFO] Ruang warna input  : RGB → dikonversi ke Lab")
print("-" * 58)

# Konversi ke Lab
image_lab = rgb2lab(image_astro_float)
rows, cols, dims = image_lab.shape
pixel_features = image_lab.reshape(rows * cols, dims)
print(f"[LAB]  Shape fitur piksel : {pixel_features.shape}  (piksel x dimensi Lab)")
print("-" * 58)

# K-Means dengan K=4
n_clusters = 4
print(f"[KMEANS] Jumlah klaster (K) : {n_clusters}")
print(f"[KMEANS] n_init             : 10 (dijalankan 10x, diambil hasil terbaik)")

kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pixel_labels = kmeans.fit_predict(pixel_features)

print(f"[KMEANS] Inertia (SSE)      : {kmeans.inertia_:.2f}  (semakin kecil semakin baik)")
print("-" * 58)
print(f"{'Klaster':<10} {'Jumlah Piksel':<18} {'% Area':<10} {'Pusat Lab (L,a,b)'}")
print("-" * 58)

for k in range(n_clusters):
    n_px = (pixel_labels == k).sum()
    pct  = n_px / len(pixel_labels) * 100
    center = kmeans.cluster_centers_[k]
    print(f"  K={k}     {n_px:<18} {pct:<10.2f}  L={center[0]:.1f}, a={center[1]:.1f}, b={center[2]:.1f}")

print("=" * 58)

# Buat citra hasil segmentasi
segmented_labels = pixel_labels.reshape(rows, cols)
segmented_image_kmeans = np.zeros_like(image_lab)
centers_lab = kmeans.cluster_centers_
for k in range(n_clusters):
    mask_k = (pixel_labels == k).reshape(rows, cols)
    segmented_image_kmeans[mask_k] = centers_lab[k]
segmented_image_rgb = lab2rgb(segmented_image_kmeans)

# Visualisasi
fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image_astro)
ax[0].set_title('Original Image (RGB)')
ax[0].axis('off')

ax[1].imshow(segmented_labels, cmap='viridis')
ax[1].set_title(f'K-Means Labels (K={n_clusters})')
ax[1].axis('off')

ax[2].imshow(segmented_image_rgb)
ax[2].set_title(f'Segmented Image (K-Means, K={n_clusters})')
ax[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# === EKSPERIMEN: Variasi nilai K ===
k_values = [2, 3, 4, 5, 6, 8]

print("=" * 60)
print("   EKSPERIMEN – Pengaruh Nilai K pada K-Means")
print("=" * 60)
print(f"{'K':<6} {'Inertia (SSE)':<20} {'Keterangan'}")
print("-" * 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for idx, k in enumerate(k_values):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        labels = km.fit_predict(pixel_features)

    keterangan = "Sangat sederhana" if k <= 2 else ("Detail tinggi" if k >= 6 else "Seimbang")
    print(f"  {k:<5} {km.inertia_:<20.2f} {keterangan}")

    seg_img = np.zeros_like(image_lab)
    for ki in range(k):
        mask = (labels == ki).reshape(rows, cols)
        seg_img[mask] = km.cluster_centers_[ki]

    row, col = divmod(idx, 3)
    axes[row][col].imshow(lab2rgb(seg_img))
    axes[row][col].set_title(f'K-Means K={k}\n(Inertia={km.inertia_:.0f})', fontsize=9)
    axes[row][col].axis('off')

print("=" * 60)
print("[INFO] Semakin besar K → Inertia mengecil, detail semakin tinggi")
print("[INFO] Namun K terlalu besar → over-segmentation, sulit diinterpretasi")

plt.suptitle('Eksperimen K-Means – Variasi Nilai K', fontsize=12)
plt.tight_layout()
plt.show()

### Analisis Hasil Praktikum 3

**Pertanyaan:** Apakah jumlah klaster (K) mempengaruhi hasil segmentasi secara signifikan? Cobalah mengubah nilai K. Apakah penggunaan ruang warna Lab membantu?

**Jawaban:**

1. **Pengaruh nilai K terhadap hasil segmentasi:**
   Ya, nilai K berpengaruh sangat signifikan. Dari hasil eksperimen terlihat bahwa:
   - **K=2:** Segmentasi sangat kasar — hanya memisahkan area terang dan gelap secara besar-besaran. Detail warna seperti warna jas oranye dan latar belakang biru bercampur.
   - **K=3–4:** Mulai terlihat pemisahan yang lebih bermakna. Jas oranye, wajah, latar belakang, dan helm hitam mulai terbedakan.
   - **K=5–6:** Detail semakin halus, gradasi warna lebih terwakili, namun mulai muncul fragmentasi kecil yang tidak perlu.
   - **K=8:** Sangat detail tetapi terlalu banyak region kecil sehingga citra terlihat "pecah" dan sulit diinterpretasikan secara semantik.
   Selain itu, nilai Inertia (SSE) terus menurun seiring bertambahnya K, menunjukkan klaster semakin rapat.

2. **Manfaat ruang warna Lab:**
   Penggunaan ruang warna Lab (CIELAB) sangat membantu. Berbeda dengan RGB, Lab dirancang agar jarak antar warna sesuai dengan persepsi mata manusia. Komponen L mengukur kecerahan secara terpisah dari warna (a=hijau↔merah, b=biru↔kuning), sehingga K-Means di Lab menghasilkan klaster yang lebih natural dan perceptually uniform dibandingkan jika langsung menggunakan RGB.

---
## Praktikum 4: Segmentasi Berbasis Tepi Menggunakan Watershed

**Tujuan:** Memperkenalkan algoritma Watershed untuk memisahkan objek yang saling bersentuhan menggunakan gradien citra sebagai input topografi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, filters, segmentation, measure
from scipy import ndimage as ndi

image_coins = data.coins()

print("=" * 58)
print("   PRAKTIKUM 4 – Watershed Segmentation")
print("=" * 58)
print(f"[INFO] Citra dimuat  : data.coins()")
print(f"[INFO] Ukuran citra  : {image_coins.shape}")
print("-" * 58)

# Elevation map (gradien Sobel)
elevation_map = filters.sobel(image_coins)
print(f"[SOBEL] Gradien min : {elevation_map.min():.4f}")
print(f"[SOBEL] Gradien max : {elevation_map.max():.4f}")
print(f"[SOBEL] Gradien mean: {elevation_map.mean():.4f}")
print("-" * 58)

# Marker
markers = np.zeros_like(image_coins)
markers[image_coins < 30]  = 1   # background
markers[image_coins > 150] = 2   # objek (koin)

n_background = (markers == 1).sum()
n_objek      = (markers == 2).sum()
n_unlabeled  = (markers == 0).sum()

print(f"[MARKER] Piksel background (label=1) : {n_background}  (intensitas < 30)")
print(f"[MARKER] Piksel objek/koin (label=2) : {n_objek}  (intensitas > 150)")
print(f"[MARKER] Piksel unlabeled  (label=0) : {n_unlabeled}  (akan ditentukan Watershed)")
print("-" * 58)

# Watershed
segmentation_watershed = segmentation.watershed(elevation_map, markers)
unique_labels = np.unique(segmentation_watershed)
print(f"[WATERSHED] Label unik hasil segmentasi : {unique_labels}")
for lbl in unique_labels:
    n = (segmentation_watershed == lbl).sum()
    nama = 'Background' if lbl == 1 else 'Objek (Koin)'
    print(f"[WATERSHED] Label {lbl} ({nama}) : {n} piksel ({n/image_coins.size*100:.1f}%)")
print("=" * 58)

segmented_colored = segmentation.mark_boundaries(image_coins, segmentation_watershed)

fig, axes = plt.subplots(ncols=4, figsize=(16, 4), sharex=True, sharey=True)
ax = axes.ravel()

ax[0].imshow(image_coins, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(elevation_map, cmap=plt.cm.nipy_spectral)
ax[1].set_title('Elevation Map\n(Sobel Gradient)')
ax[1].axis('off')

ax[2].imshow(markers, cmap=plt.cm.nipy_spectral)
ax[2].set_title(f'Markers\n(BG={n_background}px, Obj={n_objek}px)')
ax[2].axis('off')

ax[3].imshow(segmented_colored)
ax[3].set_title('Watershed Segmentation\n(batas kuning)')
ax[3].axis('off')

plt.tight_layout()
plt.show()

### Analisis Hasil Praktikum 4

**Pertanyaan:** Bagaimana pemilihan marker mempengaruhi hasil akhir? Apa yang terjadi jika marker tidak tepat?

**Jawaban:**

Algoritma Watershed bekerja dengan analogi topografi: citra gradien dianggap sebagai peta ketinggian, dan algoritma "mengalirkan air" dari marker yang sudah ditentukan ke atas. Batas antara dua aliran air itulah yang menjadi batas segmentasi.

1. **Peran elevation map (Sobel Gradient):**
   Filter Sobel menghitung gradien intensitas di setiap piksel. Tepi objek memiliki gradien tinggi (nilai besar), sedangkan interior objek memiliki gradien rendah. Inilah yang menjadi "pegunungan" dan "lembah" dalam analogi topografi.

2. **Pengaruh pemilihan marker:**
   Marker sangat krusial. Dalam praktikum ini, marker ditentukan berdasarkan intensitas:
   - Piksel sangat gelap (< 30) → latar belakang
   - Piksel sangat terang (> 150) → koin
   Hasilnya cukup baik karena koin memiliki nilai intensitas yang jelas berbeda dari background.

3. **Akibat marker yang tidak tepat:**
   Jika marker salah ditempatkan (misalnya area background ter-label sebagai objek), hasil Watershed akan salah total karena algoritma akan "tumbuh" dari titik yang keliru. Hal ini bisa menyebabkan objek bergabung atau background terpotong. Itulah mengapa Watershed sering dikombinasikan dengan pra-pemrosesan (denoising, morfologi) untuk mendapatkan marker yang akurat.

---
## Praktikum 5: Perbandingan Visual Hasil Segmentasi

**Tujuan:** Membandingkan hasil dari beberapa metode segmentasi pada citra yang sama.

In [ ]:
import matplotlib.pyplot as plt
from skimage import data, filters, segmentation, img_as_float
from sklearn.cluster import KMeans
import numpy as np
import warnings

image = data.camera()
image_float = img_as_float(image)

print("=" * 65)
print("   PRAKTIKUM 5 – Perbandingan Visual Hasil Segmentasi")
print("=" * 65)
print(f"[INFO] Citra yang digunakan : data.camera() | Shape: {image.shape}")
print("-" * 65)

# --- a) Otsu ---
thresh_otsu = filters.threshold_otsu(image)
binary_otsu = image > thresh_otsu
print(f"[OTSU]      Threshold otomatis : {thresh_otsu:.2f}")
print(f"[OTSU]      Piksel objek       : {binary_otsu.sum()} ({binary_otsu.sum()/image.size*100:.1f}%)")
print(f"[OTSU]      Piksel background  : {(~binary_otsu).sum()} ({(~binary_otsu).sum()/image.size*100:.1f}%)")
print("-" * 65)

# --- b) K-Means K=3 ---
rows, cols = image.shape
pixel_features = image_float.reshape(rows * cols, 1)
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pixel_labels = kmeans.fit_predict(pixel_features)
segmented_kmeans_labels = pixel_labels.reshape(rows, cols)
print(f"[K-MEANS]   Jumlah klaster (K) : {n_clusters}")
print(f"[K-MEANS]   Inertia (SSE)       : {kmeans.inertia_:.2f}")
for k in range(n_clusters):
    n_px = (pixel_labels == k).sum()
    center_intensity = kmeans.cluster_centers_[k][0]
    print(f"[K-MEANS]   Klaster {k}: {n_px} piksel ({n_px/image.size*100:.1f}%), "
          f"intensitas rata-rata={center_intensity:.3f} (~{center_intensity*255:.0f}/255)")
print("-" * 65)

# --- c) Watershed ---
elevation_map = filters.sobel(image)
markers = np.zeros_like(image)
markers[image < thresh_otsu] = 1
markers[image > thresh_otsu] = 2
segmentation_watershed = segmentation.watershed(elevation_map, markers)
n_ws_bg  = (segmentation_watershed == 1).sum()
n_ws_obj = (segmentation_watershed == 2).sum()
print(f"[WATERSHED] Marker dari threshold Otsu ({thresh_otsu:.0f})")
print(f"[WATERSHED] Piksel background : {n_ws_bg} ({n_ws_bg/image.size*100:.1f}%)")
print(f"[WATERSHED] Piksel objek      : {n_ws_obj} ({n_ws_obj/image.size*100:.1f}%)")
print("=" * 65)

# Visualisasi
segmented_watershed_colored = segmentation.mark_boundaries(
    image_float, segmentation_watershed, color=(1, 0, 0))

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
ax = axes.ravel()

ax[0].imshow(image, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(binary_otsu, cmap=plt.cm.gray)
ax[1].set_title(f'Otsu Thresholding\n(T={thresh_otsu:.0f})')
ax[1].axis('off')

ax[2].imshow(segmented_kmeans_labels, cmap='viridis')
ax[2].set_title(f'K-Means (K={n_clusters})')
ax[2].axis('off')

ax[3].imshow(segmented_watershed_colored)
ax[3].set_title('Watershed\n(batas merah)')
ax[3].axis('off')

plt.suptitle('Comparison of Segmentation Methods – data.camera()', fontsize=12)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Analisis Hasil Praktikum 5

**Pertanyaan:** Metode mana yang paling baik memisahkan objek utama? Mana yang paling detail? Apa kelebihan dan kekurangan masing-masing?

**Jawaban:**

Berdasarkan perbandingan visual dan data statistik output di atas pada citra `data.camera()`:

| Metode | Pemisahan Objek | Detail Region | Kelebihan | Kekurangan |
|---|---|---|---|---|
| **Otsu** | ✅ Baik (binary) | Rendah (hanya 2 kelas) | Cepat, otomatis, mudah | Hanya 2 kelas, sensitif noise |
| **K-Means** | ✅ Cukup (3 kelas) | Sedang | Multi-kelas, fleksibel | Perlu tentukan K, lambat |
| **Watershed** | ✅✅ Paling detail | Tinggi (batas tepi) | Presisi di tepi objek | Sensitif noise, butuh marker |

- **Otsu** sangat cepat dan efektif untuk pemisahan objek-background sederhana. Namun karena hanya menghasilkan 2 kelas (hitam/putih), detail bayangan atau gradasi hilang.
- **K-Means (K=3)** mampu membedakan tiga tingkat intensitas (gelap, sedang, terang), sehingga lebih informatif dari Otsu. Cocok untuk analisis warna atau tekstur.
- **Watershed** menghasilkan batas antar-region yang paling presisi karena berbasis gradien tepi. Namun hasilnya hanya 2 region (karena marker hanya 2), mirip Otsu, tetapi batas yang dihasilkan mengikuti tepi objek secara lebih akurat.

---
## Penugasan
### Tugas 3: Eksperimen Thresholding pada `data.page()`

In [ ]:
import matplotlib.pyplot as plt
from skimage import data, filters
import numpy as np

image_page = data.page()

print("=" * 60)
print("   TUGAS 3 – Eksperimen Thresholding pada data.page()")
print("=" * 60)
print(f"[INFO] Citra : data.page() | Shape: {image_page.shape}")
print(f"[INFO] Min intensity : {image_page.min()} | Max: {image_page.max()} | Mean: {image_page.mean():.2f}")
print("-" * 60)

# 1. Otsu
thresh_otsu_p = filters.threshold_otsu(image_page)
binary_otsu_p = image_page > thresh_otsu_p
white_otsu = binary_otsu_p.sum()
print(f"[OTSU]   Threshold : {thresh_otsu_p:.2f}")
print(f"[OTSU]   Piksel putih (objek) : {white_otsu} ({white_otsu/image_page.size*100:.1f}%)")
print("-" * 60)

# 2. Local Thresholding
block_size = 35
thresh_local_p = filters.threshold_local(image_page, block_size=block_size, offset=10)
binary_local_p = image_page > thresh_local_p
white_local = binary_local_p.sum()
print(f"[LOCAL]  Block size : {block_size} | Offset : 10")
print(f"[LOCAL]  Piksel putih (objek) : {white_local} ({white_local/image_page.size*100:.1f}%)")
print("-" * 60)

# 3. Yen
thresh_yen_p = filters.threshold_yen(image_page)
binary_yen_p = image_page > thresh_yen_p
white_yen = binary_yen_p.sum()
print(f"[YEN]    Threshold : {thresh_yen_p:.2f}")
print(f"[YEN]    Piksel putih (objek) : {white_yen} ({white_yen/image_page.size*100:.1f}%)")
print("=" * 60)
print("[SUMMARY] Perbandingan:")
print(f"  - Otsu   T={thresh_otsu_p:.1f}  → {white_otsu/image_page.size*100:.1f}% putih")
print(f"  - Yen    T={thresh_yen_p:.1f}  → {white_yen/image_page.size*100:.1f}% putih")
print(f"  - Local  (adaptif per blok) → {white_local/image_page.size*100:.1f}% putih")
print("=" * 60)

# Visualisasi
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax = axes.ravel()

ax[0].imshow(image_page, cmap=plt.cm.gray)
ax[0].set_title('Original Image (data.page())')
ax[0].axis('off')

ax[1].imshow(binary_otsu_p, cmap=plt.cm.gray)
ax[1].set_title(f'Otsu (T={thresh_otsu_p:.1f})\n{white_otsu/image_page.size*100:.1f}% area putih')
ax[1].axis('off')

ax[2].imshow(binary_local_p, cmap=plt.cm.gray)
ax[2].set_title(f'Local Threshold (block={block_size})\n{white_local/image_page.size*100:.1f}% area putih')
ax[2].axis('off')

ax[3].imshow(binary_yen_p, cmap=plt.cm.gray)
ax[3].set_title(f'Yen (T={thresh_yen_p:.1f})\n{white_yen/image_page.size*100:.1f}% area putih')
ax[3].axis('off')

plt.suptitle('Tugas 3 – Perbandingan Thresholding pada data.page()', fontsize=12)
plt.tight_layout()
plt.show()

### Analisis Tugas 3

**Pertanyaan:** Apakah Otsu memuaskan untuk `data.page()`? Mengapa? Bandingkan dengan `threshold_local` dan `threshold_yen`.

**Jawaban:**

Citra `data.page()` adalah citra teks dokumen yang memiliki pencahayaan tidak merata — sebagian area terlihat lebih gelap/terang akibat kondisi scan.

- **Otsu:** Hasilnya **kurang memuaskan** untuk citra ini. Karena Otsu menggunakan satu nilai ambang global untuk seluruh citra, area dengan pencahayaan bervariasi menyebabkan teks di bagian gelap ikut hilang (false negative) dan background di bagian terang ikut terdeteksi sebagai objek (false positive). Histogram citra ini tidak bersifat bimodal yang sempurna.

- **Local Thresholding (Adaptif):** Memberikan hasil **terbaik** untuk citra dokumen ini. Dengan menghitung threshold secara lokal per blok piksel, metode ini mampu menyesuaikan diri dengan variasi pencahayaan. Teks tetap terbaca dengan jelas di seluruh area citra.

- **Yen:** Menghasilkan threshold yang lebih tinggi dari Otsu, sehingga lebih banyak piksel yang diklasifikasikan sebagai background. Pada citra teks yang gelap, ini bisa menyebabkan sebagian teks hilang. Yen cocok untuk citra dengan distribusi intensitas sangat tidak seimbang.

**Kesimpulan:** Untuk citra dokumen dengan pencahayaan tidak merata, `threshold_local` adalah pilihan terbaik karena bersifat adaptif.

---
### Tugas 4: Segmentasi pada Citra dari Internet

Terapkan tiga metode segmentasi (Otsu, K-Means, Watershed) pada citra pilihan dari internet.

In [ ]:
from skimage import io as skio
import matplotlib.pyplot as plt
import numpy as np

# Citra buah apel dari internet
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Red_Apple.jpg/800px-Red_Apple.jpg"

print("=" * 55)
print("   TUGAS 4 – Segmentasi Citra dari Internet")
print("=" * 55)
print(f"[INFO] Memuat citra dari URL...")
print(f"[INFO] URL: {image_url}")

try:
    image_custom = skio.imread(image_url)
    print(f"[OK]   Citra berhasil dimuat!")
    print(f"[INFO] Shape : {image_custom.shape}")
    print(f"[INFO] Dtype : {image_custom.dtype}")
    print(f"[INFO] Jenis : {'RGB (berwarna)' if image_custom.ndim == 3 else 'Grayscale'}")
    plt.figure(figsize=(5, 4))
    plt.imshow(image_custom)
    plt.title('Citra Asli dari Internet (Buah Apel)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"[ERROR] Gagal memuat dari URL: {e}")
    print("[INFO]  Menggunakan citra alternatif dari skimage...")
    from skimage import data
    image_custom = data.coffee()
    print(f"[OK]   Menggunakan data.coffee() sebagai pengganti")
    print(f"[INFO] Shape : {image_custom.shape}")
    plt.figure(figsize=(5, 4))
    plt.imshow(image_custom)
    plt.title('Citra Alternatif: data.coffee()')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("=" * 55)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, segmentation, img_as_float
from skimage.color import rgb2gray, rgb2lab, lab2rgb
from sklearn.cluster import KMeans
import warnings

print("=" * 65)
print("   TUGAS 4 – Menerapkan 3 Metode Segmentasi")
print("=" * 65)

# Persiapan
if image_custom.ndim == 3:
    image_gray = rgb2gray(image_custom)
else:
    image_gray = image_custom.astype(float) / 255.0

image_gray_uint8 = (image_gray * 255).astype(np.uint8)
image_float_c    = img_as_float(image_gray_uint8)

print(f"[INFO] Citra grayscale shape : {image_gray_uint8.shape}")
print(f"[INFO] Rentang intensitas    : {image_gray_uint8.min()} – {image_gray_uint8.max()}")
print("-" * 65)

# ---- 1. Otsu ----
thresh_otsu_t4 = filters.threshold_otsu(image_gray_uint8)
binary_otsu_t4 = image_gray_uint8 > thresh_otsu_t4
print(f"[OTSU]      Threshold : {thresh_otsu_t4:.2f}")
print(f"[OTSU]      Objek     : {binary_otsu_t4.sum()} piksel ({binary_otsu_t4.sum()/image_gray_uint8.size*100:.1f}%)")
print("-" * 65)

# ---- 2. K-Means ----
if image_custom.ndim == 3:
    img_float_rgb_t4 = image_custom.astype(float) / 255.0
    img_lab_t4 = rgb2lab(img_float_rgb_t4)
    r4, c4, d4 = img_lab_t4.shape
    pf_t4 = img_lab_t4.reshape(r4 * c4, d4)
else:
    r4, c4 = image_gray_uint8.shape
    pf_t4 = image_float_c.reshape(r4 * c4, 1)

n_k4 = 4
km_t4 = KMeans(n_clusters=n_k4, random_state=0, n_init=10)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lbl_t4 = km_t4.fit_predict(pf_t4)
seg_lbl_t4 = lbl_t4.reshape(r4, c4)
print(f"[K-MEANS]   K={n_k4} | Inertia: {km_t4.inertia_:.2f}")
for k in range(n_k4):
    n_px = (lbl_t4 == k).sum()
    print(f"[K-MEANS]   Klaster {k}: {n_px} piksel ({n_px/len(lbl_t4)*100:.1f}%)")
print("-" * 65)

# ---- 3. Watershed ----
elevation_t4 = filters.sobel(image_gray_uint8)
markers_t4   = np.zeros_like(image_gray_uint8)
markers_t4[image_gray_uint8 < thresh_otsu_t4] = 1
markers_t4[image_gray_uint8 > thresh_otsu_t4] = 2
seg_ws_t4 = segmentation.watershed(elevation_t4, markers_t4)
seg_ws_colored_t4 = segmentation.mark_boundaries(image_float_c, seg_ws_t4, color=(1, 0, 0))
print(f"[WATERSHED] Background : {(seg_ws_t4==1).sum()} piksel ({(seg_ws_t4==1).sum()/seg_ws_t4.size*100:.1f}%)")
print(f"[WATERSHED] Objek      : {(seg_ws_t4==2).sum()} piksel ({(seg_ws_t4==2).sum()/seg_ws_t4.size*100:.1f}%)")
print("=" * 65)

# Visualisasi
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
ax = axes.ravel()

if image_custom.ndim == 3:
    ax[0].imshow(image_custom)
else:
    ax[0].imshow(image_custom, cmap='gray')
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(binary_otsu_t4, cmap='gray')
ax[1].set_title(f'Otsu (T={thresh_otsu_t4:.1f})')
ax[1].axis('off')

ax[2].imshow(seg_lbl_t4, cmap='viridis')
ax[2].set_title(f'K-Means (K={n_k4})')
ax[2].axis('off')

ax[3].imshow(seg_ws_colored_t4)
ax[3].set_title('Watershed')
ax[3].axis('off')

plt.suptitle('Tugas 4 – Perbandingan Segmentasi pada Citra Buah Apel', fontsize=12)
plt.tight_layout()
plt.show()

### Analisis Tugas 4

**Pertanyaan:** Bandingkan ketiga metode pada citra pilihanmu. Metode mana yang paling cocok? Jelaskan alasannya!

**Jawaban:**

Citra yang digunakan adalah **buah apel merah** dengan latar belakang putih/terang yang relatif polos. Karakteristik ini sangat mempengaruhi performa tiap metode:

- **Otsu:** Bekerja dengan **sangat baik** di sini. Karena citra memiliki dua area kontras jelas — apel merah (gelap jika dikonversi ke grayscale) dan background putih — histogram bersifat bimodal sehingga Otsu menemukan threshold yang tepat. Apel terpisah rapi dari background.

- **K-Means (K=4):** Juga bekerja dengan baik dan memberikan informasi lebih kaya. Dengan 4 klaster, K-Means mampu membedakan area merah tua, merah muda, bayangan, dan background secara terpisah. Ini berguna jika kita ingin menganalisis gradasi warna buah.

- **Watershed:** Menghasilkan batas tepi yang akurat mengikuti kontur buah, namun karena hanya menggunakan 2 marker (dari Otsu), hasilnya mirip dengan Otsu dalam hal jumlah region. Kelebihan Watershed adalah batas yang dihasilkan lebih mengikuti tepi asli objek.

**Metode paling cocok: Otsu**, karena citra memiliki background polos dan kontras yang tinggi antara objek dan latar belakang, sehingga asumsi bimodal histogram terpenuhi. Otsu memberikan hasil bersih dengan komputasi paling cepat. Jika diperlukan analisis warna lebih dalam, K-Means menjadi pilihan terbaik berikutnya.

---
## Kesimpulan

Dari seluruh praktikum yang telah dilakukan pada Jobsheet 04 ini, dapat disimpulkan bahwa:

1. **Segmentasi citra** adalah proses fundamental yang bertujuan membagi citra menjadi region-region bermakna berdasarkan kesamaan properti piksel (intensitas, warna, atau tekstur).

2. **Thresholding (Otsu)** merupakan metode yang paling sederhana dan cepat. Sangat efektif untuk citra dengan kontras tinggi dan histogram bimodal, namun tidak cocok untuk citra dengan pencahayaan tidak merata (di mana `threshold_local` lebih unggul).

3. **Region Growing (Flood Fill)** bekerja secara intuitif dengan menumbuhkan region dari titik seed. Hasilnya sangat bergantung pada lokasi seed dan nilai tolerance. Tolerance besar menghasilkan region luas, tolerance kecil menghasilkan region sempit.

4. **K-Means Clustering** mampu menghasilkan segmentasi multi-kelas yang fleksibel. Penggunaan ruang warna Lab meningkatkan kualitas segmentasi karena lebih sesuai persepsi manusia. Nilai K harus dipilih dengan hati-hati — terlalu kecil menghasilkan over-simplifikasi, terlalu besar menghasilkan over-segmentation.

5. **Watershed** paling unggul dalam memisahkan objek yang saling bersentuhan dengan menggunakan gradien citra sebagai topografi. Kunci keberhasilannya terletak pada pemilihan marker yang tepat dan pra-pemrosesan yang baik.

6. Pemilihan metode segmentasi yang tepat harus disesuaikan dengan karakteristik citra dan tujuan aplikasi. Tidak ada metode tunggal yang terbaik untuk semua kasus.